# USGS splib07b — remaining chapters as `HyFourier` libraries

Builds **`FourierArchive`** libraries for USGS Spectral Library Version 7 chapters:

| Chapter | Directory | Output |
|---------|-----------|--------|
| A | Artificial Materials | `usgs_artificial.fda` |
| C | Coatings | `usgs_coatings.fda` |
| L | Liquids | `usgs_liquids.fda` |
| O | Organic Compounds | `usgs_organics.fda` |
| S | Soils and Mixtures | `usgs_soils.fda` |

For each chapter, spectra are split by **acquisition instrument** (wavelength grid / band count),
assigned **material groups** from the sample name in each filename, and written to
`public/libraries/` with entries added to `public/libraries/index.json` for iSpec.

Set `USGS_SPLIB_ROOT` if your splib07b ASCII tree is not at the default path below.

In [1]:
import glob
import json
import os
import re
from collections import defaultdict

import numpy as np
import matplotlib.pyplot as plt

import hylite
from hylite import HyLibrary
from hylite.analyse.fourier import HyFourier, FourierArchive, FOURIER_ARCHIVE_EXTENSION

USGS_ROOT = os.environ.get(
    'USGS_SPLIB_ROOT',
    '/Users/thiele67/Documents/data/Libraries/Spectra USGS/ASCIIdata_splib07b',
)
REPO_ROOT = os.path.abspath(os.path.join(os.path.dirname(hylite.__file__), '..', '..', 'ispec'))
OUT_DIR = os.path.join(REPO_ROOT, 'public', 'libraries')
INDEX_PATH = os.path.join(OUT_DIR, 'index.json')
os.makedirs(OUT_DIR, exist_ok=True)

NAN_VALUE = -1.23e34
USGS_SOURCE = 'https://www.usgs.gov/data/usgs-spectral-library-version-7-data'

CHAPTERS = [
    {
        'id': 'usgs_artificial',
        'letter': 'A',
        'dirname': 'ChapterA_ArtificialMaterials',
        'title': 'Artificial Materials',
        'description': (
            'Artificial and man-made material spectra from the USGS Spectroscopy Lab '
            '(plastics, paints, metals, etc.).'
        ),
        'test_query': 'plastic',
    },
    {
        'id': 'usgs_coatings',
        'letter': 'C',
        'dirname': 'ChapterC_Coatings',
        'title': 'Coatings',
        'description': 'Surface coating spectra (desert varnish, iron oxides, jarosite films, etc.).',
        'test_query': 'varnish',
    },
    {
        'id': 'usgs_liquids',
        'letter': 'L',
        'dirname': 'ChapterL_Liquids',
        'title': 'Liquids',
        'description': 'Liquid and frozen liquid spectra (water, ice, seawater, melting snow, etc.).',
        'test_query': 'water',
    },
    {
        'id': 'usgs_organics',
        'letter': 'O',
        'dirname': 'ChapterO_OrganicCompounds',
        'title': 'Organic Compounds',
        'description': 'Organic compound spectra from the USGS Spectroscopy Lab.',
        'test_query': 'naphthalene',
    },
    {
        'id': 'usgs_soils',
        'letter': 'S',
        'dirname': 'ChapterS_SoilsAndMixtures',
        'title': 'Soils and Mixtures',
        'description': 'Soil, regolith, and mineral mixture spectra from the USGS Spectroscopy Lab.',
        'test_query': 'kaolin',
    },
]

print('USGS root:', USGS_ROOT)
print('Output dir:', OUT_DIR)

USGS root: /Users/thiele67/Documents/data/Libraries/Spectra USGS/ASCIIdata_splib07b
Output dir: /Users/thiele67/Documents/python/ispec/public/libraries


## 1. Wavelength grids and instrument keys

Each spectrum band count maps to one of the splib07b `*Wavelengths*.txt` companion files.

In [2]:
wav_by_n = {}
wav_label = {}
for path in glob.glob(os.path.join(USGS_ROOT, '*Wavelengths*.txt')):
    wav = np.loadtxt(path, skiprows=1, dtype=np.float64) * 1000.0
    wav_by_n[len(wav)] = wav
    wav_label[len(wav)] = os.path.basename(path)


def instrument_key(n_bands):
    label = wav_label[n_bands].lower()
    if 'beck' in label:
        return 'beck'
    if 'asdf' in label or 'asdhr' in label or 'asdng' in label:
        return 'asdf'
    if 'aviris' in label:
        return 'aviris'
    if 'nic4' in label:
        return 'nic4'
    return f'n{n_bands}'


def make_instrument(n_bands):
    wav = wav_by_n[n_bands]
    return {
        'key': instrument_key(n_bands),
        'n_bands': n_bands,
        'wav': wav,
        'label': wav_label[n_bands],
        'range_nm': (float(wav[0]), float(wav[-1])),
    }

## 2. Sample names and material groups

Groups use the **first name token** after stripping the splib07b prefix and instrument suffix
(e.g. `plastic`, `kaolin`, `naphthalene`).

In [3]:
def load_usgs_spectrum(path):
    refl = np.loadtxt(path, skiprows=1, dtype=np.float32)
    refl[refl == NAN_VALUE] = np.nan
    return refl


def material_stem_from_path(path):
    base = os.path.splitext(os.path.basename(path))[0]
    if base.startswith('splib07b_'):
        base = base[len('splib07b_'):]
    parts = base.split('_')
    while parts:
        token = parts[-1]
        if token.upper() in ('AREF', 'RREF'):
            parts.pop()
            continue
        if re.match(r'^(BECK|ASD|NIC|AVIRIS)', token, re.I):
            parts.pop()
            continue
        break
    return '_'.join(parts).lower() if parts else base.lower()


def group_key_from_path(path):
    stem = material_stem_from_path(path)
    token = stem.split('_')[0]
    return token if token else 'unclassified'


def feature_wavelength_range(inst):
    wav = inst['wav']
    lo = max(400.0, float(wav[0]))
    hi = min(15000.0, float(wav[-1]))
    return lo, hi

## 3. Build one chapter archive

In [4]:
def discover_instruments(chapter_dir):
    paths_by_n = defaultdict(list)
    for path in sorted(glob.glob(os.path.join(chapter_dir, '*.txt'))):
        n_bands = len(load_usgs_spectrum(path))
        if n_bands not in wav_by_n:
            raise ValueError(f'No wavelength file for {n_bands} bands: {path}')
        paths_by_n[n_bands].append(path)

    instruments = [
        make_instrument(n_bands)
        for n_bands in sorted(paths_by_n.keys(), key=lambda n: wav_by_n[n][0])
    ]
    for inst in instruments:
        inst['paths'] = paths_by_n[inst['n_bands']]
    return instruments


def build_instrument_library(inst):
    names, spectra = [], []
    for path in inst['paths']:
        names.append(os.path.splitext(os.path.basename(path))[0])
        spectra.append(load_usgs_spectrum(path))
    data = np.stack(spectra, axis=0)[:, None, :]
    lib = HyLibrary(data, lab=names, wav=inst['wav'])

    group_ids = defaultdict(list)
    for index, path in enumerate(inst['paths']):
        group_ids[group_key_from_path(path)].append(index)
    for group, ids in sorted(group_ids.items()):
        lib.add_group(group, ids)
    return lib


def build_chapter_archive(chapter):
    chapter_dir = os.path.join(USGS_ROOT, chapter['dirname'])
    if not os.path.isdir(chapter_dir):
        raise FileNotFoundError(f'Missing chapter directory: {chapter_dir}')

    instruments = discover_instruments(chapter_dir)
    archive = FourierArchive()

    print(f"\n=== Chapter {chapter['letter']}: {chapter['title']} ===")
    print('Instruments:')
    for inst in instruments:
        print(
            f"  {inst['key']:6s}  n={inst['n_bands']:4d}  "
            f"{inst['range_nm'][0]:8.1f}-{inst['range_nm'][-1]:8.1f} nm  "
            f"{len(inst['paths'])} spectra"
        )

    for inst in instruments:
        key = inst['key']
        print(f'Building {chapter["id"]} / {key} ...')
        lib = build_instrument_library(inst)
        lo, hi = feature_wavelength_range(inst)
        subset = lib.export_bands((lo, hi))
        hf = HyFourier(subset, padding='cosine', max_freq=0.25, vb=True)
        hf.precomputeExtrema(kde_sigma=10.0, vb=True)
        archive[key] = hf
        print(
            f'  {lib.sample_count():4d} samples, {len(lib.get_groups()):3d} groups, '
            f'HyFourier ({lo:.0f}-{hi:.0f} nm) {hf.data.shape}'
        )

    return archive

## 4. Build all chapters and save `.fda` files

In [5]:
built = {}
for chapter in CHAPTERS:
    archive = build_chapter_archive(chapter)
    fda_path = os.path.join(OUT_DIR, chapter['id'])
    archive.save(fda_path)
    built[chapter['id']] = fda_path + FOURIER_ARCHIVE_EXTENSION
    print('Saved', built[chapter['id']])


=== Chapter A: Artificial Materials ===
Instruments:
  beck    n=3961     205.0-  2976.0 nm  20 spectra
  asdf    n=2151     350.0-  2500.0 nm  263 spectra
  nic4    n=4595    1122.6-216006.0 nm  7 spectra
Building usgs_artificial / beck ...


    20 samples,  15 groups, HyFourier (400-2976 nm) (20, 845, 2)
Building usgs_artificial / asdf ...


   263 samples,  92 groups, HyFourier (400-2500 nm) (263, 525, 2)
Building usgs_artificial / nic4 ...


     7 samples,   5 groups, HyFourier (1123-15000 nm) (7, 1069, 2)
Saved /Users/thiele67/Documents/python/ispec/public/libraries/usgs_artificial.fda

=== Chapter C: Coatings ===
Instruments:
  beck    n=3961     205.0-  2976.0 nm  12 spectra
Building usgs_coatings / beck ...


    12 samples,   5 groups, HyFourier (400-2976 nm) (12, 845, 2)
Saved /Users/thiele67/Documents/python/ispec/public/libraries/usgs_coatings.fda

=== Chapter L: Liquids ===
Instruments:
  beck    n=3961     205.0-  2976.0 nm  3 spectra
  asdf    n=2151     350.0-  2500.0 nm  21 spectra
Building usgs_liquids / beck ...


...

   142 samples, 104 groups, HyFourier (400-2500 nm) (142, 525, 2)
Building usgs_organics / nic4 ...


   218 samples,  99 groups, HyFourier (1123-15000 nm) (218, 1069, 2)
Saved /Users/thiele67/Documents/python/ispec/public/libraries/usgs_organics.fda

=== Chapter S: Soils and Mixtures ===
Instruments:
  beck    n=3961     205.0-  2976.0 nm  73 spectra
  asdf    n=2151     350.0-  2500.0 nm  103 spectra
  nic4    n=4595    1122.6-216006.0 nm  33 spectra
Building usgs_soils / beck ...


    73 samples,  67 groups, HyFourier (400-2976 nm) (73, 845, 2)
Building usgs_soils / asdf ...


   103 samples,  71 groups, HyFourier (400-2500 nm) (103, 525, 2)
Building usgs_soils / nic4 ...


    33 samples,  31 groups, HyFourier (1123-15000 nm) (33, 1069, 2)
Saved /Users/thiele67/Documents/python/ispec/public/libraries/usgs_soils.fda


## 5. Register libraries in `index.json`

In [6]:
with open(INDEX_PATH, 'r', encoding='utf-8') as handle:
    catalog = json.load(handle)

existing_ids = {entry['id'] for entry in catalog.get('libraries', [])}

for chapter in CHAPTERS:
    entry = {
        'id': chapter['id'],
        'name': f"USGS Spectral Library (V7) Chapter {chapter['letter']}: {chapter['title']}",
        'source': USGS_SOURCE,
        'description': chapter['description'],
        'file': f"{chapter['id']}.fda",
        'default': False,
    }
    if chapter['id'] in existing_ids:
        catalog['libraries'] = [
            entry if item['id'] == chapter['id'] else item
            for item in catalog['libraries']
        ]
    else:
        catalog['libraries'].append(entry)

with open(INDEX_PATH, 'w', encoding='utf-8') as handle:
    json.dump(catalog, handle, indent=2)
    handle.write('\n')

print('Updated', INDEX_PATH)
for entry in catalog['libraries']:
    if entry['id'].startswith('usgs_'):
        print(' ', entry['id'], '→', entry['file'])

Updated /Users/thiele67/Documents/python/ispec/public/libraries/index.json
  usgs_minerals → usgs_minerals.fda
  usgs_vegetation → usgs_vegetation.fda
  usgs_artificial → usgs_artificial.fda
  usgs_coatings → usgs_coatings.fda
  usgs_liquids → usgs_liquids.fda
  usgs_organics → usgs_organics.fda
  usgs_soils → usgs_soils.fda


## 6. Smoke-test search on each new library

In [7]:
for chapter in CHAPTERS:
    archive = FourierArchive.load(os.path.join(OUT_DIR, chapter['id']))
    query = chapter['test_query']
    names, scores = archive.search(query, confidence=20.0, n_result=5)
    print(f"\n{chapter['id']} — search {query!r} (instruments: {list(archive.keys())})")
    for name, score in zip(names, scores):
        print(f'  {score:.4f} {name}')


usgs_artificial — search 'plastic' (instruments: ['beck', 'asdf', 'nic4'])
  1.0000 (asdf) [plastic] splib07b_Plastic_HDPE_GDS400_WhTrnslu_ASDFRa_AREF
  1.0000 (asdf) [plastic] splib07b_Plastic_HDPE_GDS394_LgryTrns_ASDFRa_AREF
  1.0000 (asdf) [plastic] splib07b_Plastic_HDPE_GDS393_GlosWhTr_ASDFRa_AREF
  1.0000 (asdf) [plastic] splib07b_Plastic_PETE_GDS383_Clrbluis_ASDFRa_AREF
  1.0000 (asdf) [plastic] splib07b_Plastic_ABS_GDS341_BlackPipe_ASDFRa_AREF

usgs_coatings — search 'varnish' (instruments: ['beck'])
  1.0000 (beck) [desert] splib07b_Desert_Varnish_ANP90-14_BECKa_AREF
  1.0000 (beck) [desert] splib07b_Desert_Varnish_GDS141_BECKa_AREF
  1.0000 (beck) [desert] splib07b_Desert_Varnish_GDS78A_Rhy_BECKa_AREF
  0.0000 (beck) [jarosite] splib07b_Jarosite_Thin_Film_GDS243_BECKb_AREF
  0.0000 (beck) [jarosite] splib07b_Jarosite_on_Qtzite_BR93-34A2_BECKa_AREF

usgs_liquids — search 'water' (instruments: ['beck', 'asdf'])
  1.0000 (asdf) [water+montmor] splib07b_Water+Montmor_SWy-2+16.5g-